# Safety Helmet Detection - Kaggle Training Notebook
## YOLOv8m Fine-Tuning

**Dataset**: Safety Helmet (RoboFlow/Kaggle input) | **Model**: YOLOv8-Medium | **Classes**: 1 (safetyhelmet)

Add the Safety Helmet YOLOv8 dataset to the Kaggle notebook with **Add Input** before running.

## 1. Environment Setup

In Kaggle, enable GPU acceleration and Internet before running this cell.

In [ ]:
!nvidia-smi
!pip install ultralytics -q

In [ ]:
from ultralytics import YOLO
import torch
import os
import yaml
import glob
import shutil
import zipfile
from pathlib import Path
from collections import Counter
from IPython.display import Image as IPImage, display

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f"VRAM: {vram / 1e9:.1f} GB")

## 2. Kaggle Dataset Paths

This notebook expects the dataset to be attached as a Kaggle input. Kaggle mounts inputs under `/kaggle/input` and saves generated files under `/kaggle/working`.

In [ ]:
KAGGLE_INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = "/kaggle/working"
DATA_YAML = os.path.join(OUTPUT_DIR, "data_helmet.yaml")

def looks_like_helmet_dataset(data_yaml_path):
    dataset_dir = data_yaml_path.parent
    try:
        with open(data_yaml_path, "r") as f:
            config = yaml.safe_load(f) or {}
    except Exception:
        return False

    names = config.get("names", [])
    if isinstance(names, dict):
        names = list(names.values())
    name_text = " ".join(str(name).lower() for name in names)

    has_yolo_splits = (
        (dataset_dir / "train" / "images").exists()
        and (dataset_dir / "valid" / "images").exists()
    )
    path_text = str(dataset_dir).lower()
    return has_yolo_splits and ("helmet" in name_text or "hardhat" in name_text or "helmet" in path_text)

candidate_yamls = list(KAGGLE_INPUT_DIR.rglob("data.yaml")) if KAGGLE_INPUT_DIR.exists() else []
matching_yamls = [path for path in candidate_yamls if looks_like_helmet_dataset(path)]

if not matching_yamls:
    helmet_zips = [path for path in KAGGLE_INPUT_DIR.rglob("*.zip") if "helmet" in path.name.lower()]
    if helmet_zips:
        extract_dir = Path(OUTPUT_DIR) / "safety_helmet_dataset"
        extract_dir.mkdir(parents=True, exist_ok=True)
        print(f"Extracting {helmet_zips[0]} to {extract_dir}...")
        with zipfile.ZipFile(helmet_zips[0], "r") as zip_ref:
            zip_ref.extractall(extract_dir)
        extracted_yamls = list(extract_dir.rglob("data.yaml"))
        candidate_yamls.extend(extracted_yamls)
        matching_yamls = [path for path in extracted_yamls if looks_like_helmet_dataset(path)]

if matching_yamls:
    DATASET_DIR = str(matching_yamls[0].parent)
else:
    fallback = KAGGLE_INPUT_DIR / "safety-helmet-yolov8" / "Safetyhelmet-detection.v1i.yolov8"
    if fallback.exists():
        DATASET_DIR = str(fallback)
    else:
        available = "\n".join(str(path) for path in candidate_yamls) or "No data.yaml files found."
        raise FileNotFoundError(
            "Safety helmet dataset was not found under /kaggle/input. "
            "Attach the YOLOv8 dataset with Kaggle Add Input, then rerun this cell.\n\n"
            f"Available data.yaml files:\n{available}"
        )

train_images = os.path.join(DATASET_DIR, "train", "images")
valid_images = os.path.join(DATASET_DIR, "valid", "images")

print("Environment: Kaggle")
print(f"Dataset dir: {DATASET_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Train images: {len(os.listdir(train_images))}")
print(f"Valid images: {len(os.listdir(valid_images))}")

## 3. Configuration

In [ ]:
# --- Model ---
MODEL_NAME = "yolov8m.pt"
IMAGE_SIZE = 640
EPOCHS = 100
BATCH_SIZE = 16
PATIENCE = 20
SEED = 42

print(f"Model: {MODEL_NAME}")
print(f"Image size: {IMAGE_SIZE} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE}")

## 4. Prepare Dataset Configuration

In [ ]:
with open(os.path.join(DATASET_DIR, "data.yaml"), "r") as f:
    data_config = yaml.safe_load(f)

data_config["train"] = os.path.join(DATASET_DIR, "train", "images")
data_config["val"] = os.path.join(DATASET_DIR, "valid", "images")

test_images = os.path.join(DATASET_DIR, "test", "images")
if os.path.exists(test_images):
    data_config["test"] = test_images
else:
    data_config.pop("test", None)

with open(DATA_YAML, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("data.yaml:")
print(yaml.dump(data_config, default_flow_style=False))

## 5. Dataset Analysis

In [ ]:
names_raw = data_config.get("names", {})
if isinstance(names_raw, list):
    class_names = {i: name for i, name in enumerate(names_raw)}
elif isinstance(names_raw, dict):
    class_names = {int(k): v for k, v in names_raw.items()}
else:
    class_names = {}
class_counts = Counter()
split_names = ["train", "valid"] + (["test"] if os.path.exists(os.path.join(DATASET_DIR, "test")) else [])
split_counts = {split: Counter() for split in split_names}

for split in split_names:
    label_dir = os.path.join(DATASET_DIR, split, "labels")
    image_dir = os.path.join(DATASET_DIR, split, "images")
    n_images = len(glob.glob(os.path.join(image_dir, "*")))
    for label_file in glob.glob(os.path.join(label_dir, "*.txt")):
        with open(label_file, "r") as f:
            for line in f:
                cid = int(line.strip().split()[0])
                class_counts[cid] += 1
                split_counts[split][cid] += 1
    print(f"{split}: {n_images} images")

print(f"\nTotal annotations: {sum(class_counts.values())}")
for cid, count in sorted(class_counts.items()):
    print(f"  {class_names.get(cid, cid)}: {count}")

if 0 in split_counts["valid"]:
    print(f"\nTrain/Val annotations: {split_counts['train'][0]}:{split_counts['valid'][0]}")

## 6. Train Model

In [ ]:
model = YOLO(MODEL_NAME)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    workers=4,
    seed=SEED,

    # Optimizer
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_bias_lr=0.01,

    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    erasing=0.4,

    # Strategy
    close_mosaic=10,
    amp=True,

    # Output
    project=OUTPUT_DIR,
    name="safety_helmet_training",
    exist_ok=True,
    verbose=True,
)

## 7. Training Results

In [ ]:
train_dir = os.path.join(OUTPUT_DIR, "safety_helmet_training")

plots = [
    'results.png',
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'P_curve.png',
    'R_curve.png',
    'PR_curve.png',
    'F1_curve.png',
]

for name in plots:
    path = os.path.join(train_dir, name)
    if os.path.exists(path):
        print(f"\n{'='*60}")
        print(f"  {name}")
        print(f"{'='*60}")
        display(IPImage(filename=path, width=800))

## 8. Evaluation

In [ ]:
BEST_WEIGHTS = os.path.join(OUTPUT_DIR, "safety_helmet_training", "weights", "best.pt")
best_model = YOLO(BEST_WEIGHTS)
eval_split = "test" if "test" in data_config else "val"

eval_results = best_model.val(
    data=DATA_YAML,
    split=eval_split,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    conf=0.25,
    iou=0.6,
    verbose=True,
)

print(f"\n{'='*40}")
print(f"  {eval_split.upper()} RESULTS")
print(f"{'='*40}")
print(f"  mAP50:      {eval_results.box.map50:.4f}")
print(f"  mAP50-95:   {eval_results.box.map:.4f}")
print(f"  Precision:  {eval_results.box.mp:.4f}")
print(f"  Recall:     {eval_results.box.mr:.4f}")

for i, name in eval_results.names.items():
    print(f"  {name} AP50: {eval_results.box.ap50[i]:.4f}")

## 9. Export Model

In [ ]:
output_weights = os.path.join(OUTPUT_DIR, "helmet_best.pt")
shutil.copy(BEST_WEIGHTS, output_weights)
print(f"Best weights saved to {output_weights}")
print(f"Size: {os.path.getsize(output_weights) / 1e6:.1f} MB")

## 10. Output Files

In [ ]:
print("Output files:")
print("-" * 60)
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in files:
        filepath = os.path.join(root, f)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        if size_mb > 0.1:
            print(f"  {filepath} ({size_mb:.1f} MB)")